# OC-DECLARE Evaluation

This notebook reproduces the evaluation results from the paper.
It loads pre-computed OC-DECLARE discovery and reduction results, computes complexity metrics,
and generates the figures and tables used in the paper.

**Requirements:** `pip install pandas plotly kaleido networkx`

**Data:** The result JSON files are expected in `../frontend/public/models/`.

OC-DFG and OC-PN typed connection counts (from pm4py) are included as reference constants, as the export files are too large too include here. The models were discovered using:
`ocpn = pm4py.discover_oc_petri_net(ocel)` and `ocdfg = pm4py.discover_ocdfg(ocel)` on PM4Py 2.7.19.8.

In [ ]:
%pip install pandas plotly kaleido networkx

## Configuration

In [14]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.io as pio
import plotly.colors

pio.templates.default = "plotly_white"

RESULTS_PATH = Path("../frontend/public/models")
FIGURES_PATH = Path("./figures")
FIGURES_PATH.mkdir(exist_ok=True)

DATASETS = ["P2P", "O2C", "Logistics", "BPIC2014", "BPIC2017", "BPIC2019"]

MODES = {
    "CO": "GENNone-FILTERNone",
    "PO": "GENNone-FILTERSome(20)",
    "PR": "GENSome(20)-FILTERSome(20)",
}
MODE_LABELS = {
    'CO': '$\\mathsf{CO}$',
    'PO': '$\\mathsf{PO}$',
    'PR': '$\\mathsf{PR}$',
}

STAGES = {
    "discovered": "-discovered",
    "reduced": "-reduced-lossless",
    "refined": "-refined",
}

ALL_ARC_TYPES = ["EF", "EP", "AS"]

# OC-DFG and OC-PN typed connections (discovered with pm4py)
# These are reference values for the cross-formalism comparison in the paper.
OCDFG_TYPED_CONNECTIONS = {
    "P2P": 52, "O2C": 244, "Logistics": 76,
    "BPIC2014": 16836, "BPIC2017": 412, "BPIC2019": 2708,
}
OCPN_TYPED_CONNECTIONS = {
    "P2P": 86, "O2C": 160, "Logistics": 134,
    "BPIC2014": 3394, "BPIC2017": 150, "BPIC2019": 362,
}

## Helper Functions

In [3]:
def load_json(path: Path):
    with open(path, "rt") as f:
        return json.load(f)


def result_path(dataset: str, mode_key: str, suffix: str) -> Path:
    return RESULTS_PATH / f"{dataset}-None-{mode_key}{suffix}.json"


def oi_entry_count(label: dict) -> int:
    """Count total object type associations in a constraint label (= |dom(oi)|)."""
    return len(label.get("each", [])) + len(label.get("any", [])) + len(label.get("all", []))


def compute_graph_metrics(constraints: list, summary: dict) -> dict:
    """Compute paper complexity metrics from a list of constraint dicts.

    Uses num_event_types from the summary as |A| (includes activities not in any constraint).

    Returns dict with: constraints, size, density, separability, variability,
    typed_connections (= TC, sum of |dom(oi)|), and otc (object type coupling, sum of C(k,2)).
    """
    g = nx.MultiDiGraph()
    for c in constraints:
        g.add_edge(c["from"], c["to"], arc_type=c["arc_type"])

    # Add isolated nodes for event types not appearing in any constraint
    num_event_types = summary["num_event_types"]
    missing_nodes = num_event_types - g.number_of_nodes()
    for i in range(missing_nodes):
        g.add_node(f"__isolated_{i}")

    num_constraints = g.number_of_edges()
    num_nodes = g.number_of_nodes()
    size = num_nodes + num_constraints

    # Typed connections: TC(G) = sum of |dom(oi_d)|
    typed_connections = sum(oi_entry_count(c["label"]) for c in constraints)

    # Object type coupling: OTC = sum of C(k, 2) per constraint
    otc = sum(oi_entry_count(c["label"]) * (oi_entry_count(c["label"]) - 1) / 2 for c in constraints)

    # Weakly connected components
    components = list(nx.weakly_connected_components(g))

    # Density: D(G) = max over components of |D_C| / |A_C|
    density = 0.0
    variability = 0.0
    for comp_nodes in components:
        sub = g.subgraph(comp_nodes)
        n_arcs = sub.number_of_edges()
        n_nodes = sub.number_of_nodes()
        if n_nodes > 0:
            density = max(density, n_arcs / n_nodes)
        # Constraint Variability: max entropy of arc type distribution in components
        if n_arcs > 0:
            arc_type_counts = {}
            for _, _, data in sub.edges(data=True):
                at = data["arc_type"]
                arc_type_counts[at] = arc_type_counts.get(at, 0) + 1
            entropy = 0.0
            for count in arc_type_counts.values():
                if count > 0:
                    p = count / n_arcs
                    entropy -= p * math.log(p) / math.log(len(ALL_ARC_TYPES))
            variability = max(variability, entropy)

    # Separability: Sep(G) = |Comp(G)| / S(G)
    separability = len(components) / size if size > 0 else 0.0

    return {
        "constraints": num_constraints,
        "size": size,
        "density": density,
        "separability": separability,
        "variability": variability,
        "typed_connections": typed_connections,
        "otc": otc,
    }

## 1. Load Results and Compute Metrics

In [4]:
# Load summaries and compute metrics for all datasets × modes × stages
summaries = {}  # (dataset, mode_key) -> summary dict
metrics = {}    # (dataset, mode_key, stage) -> metrics dict

for dataset in DATASETS:
    for mode_name, mode_key in MODES.items():
        # Load summary (timing + dataset properties)
        summary = load_json(result_path(dataset, mode_key, "-summary"))
        summaries[(dataset, mode_key)] = summary

        # Load constraints and compute metrics for each stage
        for stage_name, stage_suffix in STAGES.items():
            constraints = load_json(result_path(dataset, mode_key, stage_suffix))
            m = compute_graph_metrics(constraints, summary)
            metrics[(dataset, mode_key, stage_name)] = m

print(f"Loaded {len(summaries)} summaries, computed {len(metrics)} metric sets.")

Loaded 18 summaries, computed 54 metric sets.


## 2. Improvement Factors (Figure 3)

Box plot showing the percentage improvement in complexity measures after lossless transitive reduction,
aggregated over datasets. The three discovery modes (CO, PO, PR) are shown separately.

In [5]:
MEASURES = ["constraints", "size", "density", "typed_connections", "variability", "separability"]
MEASURE_LABELS = {
    "constraints": "constraints",
    "size": "size",
    "density": "density",
    "typed_connections": "typed connections",
    "variability": "variability",
    "separability": "separability",
}
# For separability, higher = simpler, so we invert the improvement direction
HIGHER_IS_BETTER = {"separability"}

# Compute improvement percentages: (before - after) / before * 100
improvements = {mk: {m: [] for m in MEASURES} for mk in MODES.values()}

for dataset in DATASETS:
    for mode_key in MODES.values():
        before = metrics[(dataset, mode_key, "discovered")]
        after = metrics[(dataset, mode_key, "reduced")]
        for measure in MEASURES:
            b = before[measure]
            a = after[measure]
            if measure in HIGHER_IS_BETTER:
                b, a = (1 - b), (1 - a)
            factor = (b - a) / b * 100 if b != 0 else 0
            improvements[mode_key][measure].append(factor)

In [ ]:
palette = plotly.colors.qualitative.Bold
measure_colors = {m: palette[i % len(palette)] for i, m in enumerate(MEASURES)}

fig = go.Figure()

fig.add_shape(
    type="line", x0=-0.5, x1=len(MEASURES) - 0.5, y0=0, y1=0,
    line=dict(color="Gray", dash="dash"),
)

for i, (mode_name, mode_key) in enumerate(MODES.items()):
    opacity = 1.0 if i == 0 else 0.3 * (3 - i)
    for measure in MEASURES:
        y_data = improvements[mode_key][measure]
        fig.add_trace(go.Box(
            y=y_data,
            x=[MEASURE_LABELS[measure]] * len(y_data),
            name=MEASURE_LABELS[measure],
            marker_color=measure_colors[measure],
            legendgroup=measure,
            showlegend=(i == 0),
            opacity=opacity,
            offsetgroup=str(i),
        ))
# Position settings for the custom legend (Top right inside plot)
key_x_start = 0.725
key_y_start = 1.14
x_offsets = [0,0.075,0.15]

for i, mode in enumerate(MODES.keys()):
    # Match the opacity logic used in the loop
    op = 1.0 if i == 0 else 0.3 * (3-i)
    # 1. The Box (Shape)
    x = key_x_start + x_offsets[i]
    y = key_y_start
    fig.add_shape(
        type="rect",
        xref="paper", yref="paper",
        x0=x,
        x1=x + 0.02,
        y0=y,
        y1=y + 0.08,
        fillcolor="gray",
        opacity=op
    )
    
    fig.add_annotation(
        x=x + 0.025,
        y=y + 0.04,
        xref="paper", yref="paper",
        text=mode,
        showarrow=False,
        xanchor="left",
        yanchor="middle",
        font=dict(size=16)
    )

fig.update_layout(
    title='Improvements after Lossless Transitive Reduction',
    yaxis_title="Improvement (%)",
    boxmode="group",
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=0.99, xanchor="left", x=0, font=dict(size=18)),
    width=900,
    height=310,
    font=dict(size=18),
    margin=dict(l=80, r=20, t=58, b=0),
)

fig.update_traces(boxmean=True)
fig.update_xaxes(visible=False)
fig.show()
fig.write_image(str(FIGURES_PATH / "improvement_factors.pdf"))
print(f"Saved to {FIGURES_PATH / 'improvement_factors.pdf'}")

Saved to figures/improvement_factors.pdf


## 3. Evaluation Table (Table 1)

Dataset properties, reduced constraint counts |red(D)| for CO/PO/PR,
and typed connections TC(red(D)) compared with OC-DFG and OC-PN.

In [7]:
rows = []
for dataset in DATASETS:
    summary = summaries[(dataset, MODES["CO"])]
    row = {
        "Dataset": dataset,
        "|ET|": summary["num_event_types"],
        "|OT|": summary["num_object_types"],
        "|E|": summary["num_events"],
        "|O|": summary["num_objects"],
    }
    for mode_name, mode_key in MODES.items():
        m = metrics[(dataset, mode_key, "reduced")]
        row[f"|D| {mode_name}"] = m["constraints"]
        row[f"TC {mode_name}"] = m["typed_connections"]
    row["TC OC-DFG"] = OCDFG_TYPED_CONNECTIONS[dataset]
    row["TC OC-PN"] = OCPN_TYPED_CONNECTIONS[dataset]
    rows.append(row)

eval_df = pd.DataFrame(rows)
eval_df

,Dataset,|ET|,|OT|,|E|,|O|,|D| CO,TC CO,|D| PO,TC PO,|D| PR,TC PR,TC OC-DFG,TC OC-PN
0,P2P,10,7,14671,9543,19,30,19,30,19,30,52,86
1,O2C,11,6,21008,10840,70,152,55,128,25,37,244,160
2,Logistics,14,7,35413,13910,42,59,35,52,34,41,76,134
3,BPIC2014,330,7,690622,228885,1974,3050,2422,3454,2573,3165,16836,3394
4,BPIC2017,26,4,1202267,106162,110,121,48,59,49,49,412,150
5,BPIC2019,42,4,1595923,330685,276,622,199,509,243,388,2708,362


In [8]:
# Generate LaTeX table
print(r"""\begin{tabular}{@{}lrrrr rrr rrrrr@{}}""")
print(r"\toprule")
print(r"\multirow{2}{*}{Dataset} &")
print(r"\multirow{2}{*}{$|\evTypeUniv|$} &")
print(r"  \multirow{2}{*}{$|\obTypeUniv|$} &")
print(r"  \multirow{2}{*}{$|E|$} &")
print(r"  \multirow{2}{*}{$|O|$} &")
print(r"  \multicolumn{3}{c}{$|\red{\mathcal{D}}|$} &")
print(r"  \multicolumn{5}{c}{Typed Connections $TC(\red{\mathcal{D}})$} \\ \cmidrule(lr){6-8} \cmidrule(lr){9-13}")
print(r" & & & & &")
print(r"  \textsf{CO} & \textsf{PO} & \textsf{PR} &")
print(r"  \textsf{CO} & \textsf{PO} & \textsf{PR} &")
print(r"  \emph{OC-DFG} & \emph{OC-PN} \\ \midrule")

for _, r in eval_df.iterrows():
    print(
        f"{r['Dataset']} & {r['|ET|']} & {r['|OT|']} & {r['|E|']} & {r['|O|']}"
        f" & {r['|D| CO']} & {r['|D| PO']} & {r['|D| PR']}"
        f" & {r['TC CO']} & {r['TC PO']} & {r['TC PR']}"
        f" & {r['TC OC-DFG']} & {r['TC OC-PN']} \\\\"
    )

print(r"\bottomrule")
print(r"\end{tabular}%")

\begin{tabular}{@{}lrrrr rrr rrrrr@{}}
\toprule
\multirow{2}{*}{Dataset} &
\multirow{2}{*}{$|\evTypeUniv|$} &
  \multirow{2}{*}{$|\obTypeUniv|$} &
  \multirow{2}{*}{$|E|$} &
  \multirow{2}{*}{$|O|$} &
  \multicolumn{3}{c}{$|\red{\mathcal{D}}|$} &
  \multicolumn{5}{c}{Typed Connections $TC(\red{\mathcal{D}})$} \\ \cmidrule(lr){6-8} \cmidrule(lr){9-13}
 & & & & &
  \textsf{CO} & \textsf{PO} & \textsf{PR} &
  \textsf{CO} & \textsf{PO} & \textsf{PR} &
  \emph{OC-DFG} & \emph{OC-PN} \\ \midrule
P2P & 10 & 7 & 14671 & 9543 & 19 & 19 & 19 & 30 & 30 & 30 & 52 & 86 \\
O2C & 11 & 6 & 21008 & 10840 & 70 & 55 & 25 & 152 & 128 & 37 & 244 & 160 \\
Logistics & 14 & 7 & 35413 & 13910 & 42 & 35 & 34 & 59 & 52 & 41 & 76 & 134 \\
BPIC2014 & 330 & 7 & 690622 & 228885 & 1974 & 2422 & 2573 & 3050 & 3454 & 3165 & 16836 & 3394 \\
BPIC2017 & 26 & 4 & 1202267 & 106162 & 110 & 48 & 49 & 121 & 59 & 49 & 412 & 150 \\
BPIC2019 & 42 & 4 & 1595923 & 330685 & 276 & 199 & 243 & 622 & 509 & 388 & 2708 & 362 \\
\bottomru

## 4. Typed Connections Comparison

Bar chart comparing typed connections across formalisms:
OC-DECLARE (discovered and after lossless reduction in CO mode), OC-DFG, and OC-PN.

In [9]:
co_key = MODES["CO"]

tc_data = []
for dataset in DATASETS:
    tc_data.append({
        "Dataset": dataset,
        "OC-DECLARE (disc.)": metrics[(dataset, co_key, "discovered")]["typed_connections"],
        "OC-DECLARE (red.)": metrics[(dataset, co_key, "reduced")]["typed_connections"],
        "OC-DFG": OCDFG_TYPED_CONNECTIONS[dataset],
        "OC-PN": OCPN_TYPED_CONNECTIONS[dataset],
    })

tc_df = pd.DataFrame(tc_data)
tc_df

,Dataset,OC-DECLARE (disc.),OC-DECLARE (red.),OC-DFG,OC-PN
0,P2P,45,30,52,86
1,O2C,387,152,244,160
2,Logistics,133,59,76,134
3,BPIC2014,14964,3050,16836,3394
4,BPIC2017,693,121,412,150
5,BPIC2019,1455,622,2708,362


In [10]:
colors = ["#636EFA", "#00CC96", "#EF553B", "#AB63FA"]
columns = ["OC-DECLARE (disc.)", "OC-DECLARE (red.)", "OC-DFG", "OC-PN"]

fig = go.Figure()
for col, color in zip(columns, colors):
    fig.add_trace(go.Bar(
        name=col,
        x=tc_df["Dataset"],
        y=tc_df[col],
        marker_color=color,
    ))

fig.update_layout(
    yaxis_title="Typed Connections",
    barmode="group",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    width=900, height=500,
    font=dict(size=14),
)
fig.update_xaxes(ticks="outside", tickwidth=2, ticklen=4, tickcolor="gray")

fig.show()
fig.write_image(str(FIGURES_PATH / "cmp-typed-connections.pdf"))
print(f"Saved to {FIGURES_PATH / 'cmp-typed-connections.pdf'}")

Saved to figures/cmp-typed-connections.pdf


## 5. Individual Metric Comparison (Before vs. After Reduction)

Bar charts for each complexity metric comparing discovered vs. reduced models (CO mode).

In [11]:
co_key = MODES["CO"]
INTEGER_MEASURES = {"constraints", "size", "typed_connections"}

for measure in MEASURES:
    fig = go.Figure()

    for stage, label, color, pattern in [
        ("discovered", "Discovered", "rgb(55, 83, 109)", ""),
        ("reduced", "Reduced", "rgb(55, 83, 109)", "/"),
    ]:
        values = [metrics[(d, co_key, stage)][measure] for d in DATASETS]
        fmt = ".0f" if measure in INTEGER_MEASURES else ".2f"
        fig.add_trace(go.Bar(
            name=label,
            x=DATASETS,
            y=values,
            marker_color=color,
            marker_pattern_shape=pattern,
            marker_pattern_solidity=0.6,
            text=[f"{v:{fmt}}" for v in values],
            textposition="outside",
        ))

    fig.update_layout(
        title=MEASURE_LABELS[measure].title(),
        yaxis_title=MEASURE_LABELS[measure].title(),
        barmode="group",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        width=900, height=500,
        font=dict(size=14),
    )
    fig.update_xaxes(ticks="outside", tickwidth=2, ticklen=4, tickcolor="gray")

    fig.show()
    fig_name = measure.replace(" ", "_")
    fig.write_image(str(FIGURES_PATH / f"{fig_name}_measures.pdf"))

## 6. Processing Durations

In [12]:
duration_rows = []
for dataset in DATASETS:
    for mode_name, mode_key in MODES.items():
        s = summaries[(dataset, mode_key)]
        disc_times = s.get("discovery_durations_seconds", [])
        red_times = s.get("reduction_duration_seconds", [])
        ref_times = s.get("refinement_duration_seconds", [])
        duration_rows.append({
            "Dataset": dataset,
            "Mode": mode_name,
            "Discovery (s)": f"{np.mean(disc_times):.2f} ± {np.std(disc_times):.2f}",
            "Reduction (s)": f"{np.mean(red_times):.4f} ± {np.std(red_times):.4f}",
            "Refinement (s)": f"{np.mean(ref_times):.4f} ± {np.std(ref_times):.4f}" if ref_times else "—",
        })

dur_df = pd.DataFrame(duration_rows)
dur_df

,Dataset,Mode,Discovery (s),Reduction (s),Refinement (s)
0,P2P,CO,0.04 ± 0.00,0.0001 ± 0.0000,0.0051 ± 0.0004
1,P2P,PO,0.04 ± 0.00,0.0001 ± 0.0000,0.0048 ± 0.0004
2,P2P,PR,0.04 ± 0.00,0.0001 ± 0.0000,0.0047 ± 0.0007
3,O2C,CO,0.85 ± 0.02,0.0006 ± 0.0000,0.0504 ± 0.0017
4,O2C,PO,0.94 ± 0.02,0.0003 ± 0.0000,0.0447 ± 0.0039
5,O2C,PR,0.10 ± 0.00,0.0001 ± 0.0000,0.0418 ± 0.0021
6,Logistics,CO,0.09 ± 0.01,0.0002 ± 0.0000,0.0103 ± 0.0019
7,Logistics,PO,0.10 ± 0.01,0.0002 ± 0.0000,0.0079 ± 0.0012
8,Logistics,PR,0.08 ± 0.01,0.0002 ± 0.0000,0.0096 ± 0.0009
9,BPIC2014,CO,12.25 ± 0.35,0.2702 ± 0.0064,1.3316 ± 0.1158


## 7. Summary Statistics

In [13]:
# Print mean improvement percentages per mode
for mode_name, mode_key in MODES.items():
    print(f"\n=== {mode_name} ===")
    for measure in MEASURES:
        vals = improvements[mode_key][measure]
        print(f"  {MEASURE_LABELS[measure]:>20s}: {np.mean(vals):6.1f}% mean improvement (range: {min(vals):.1f}% – {max(vals):.1f}%)")


=== CO ===
           constraints:   67.8% mean improvement (range: 40.6% – 84.3%)
                  size:   63.1% mean improvement (range: 31.0% – 82.1%)
               density:   67.8% mean improvement (range: 40.6% – 84.3%)
     typed connections:   61.5% mean improvement (range: 33.3% – 82.5%)
           variability:    9.4% mean improvement (range: 4.5% – 14.6%)
          separability:    0.6% mean improvement (range: 0.1% – 1.1%)

=== PO ===
           constraints:   53.1% mean improvement (range: 40.6% – 64.6%)
                  size:   46.3% mean improvement (range: 31.0% – 58.8%)
               density:   53.1% mean improvement (range: 40.6% – 64.6%)
     typed connections:   48.1% mean improvement (range: 33.3% – 55.2%)
           variability:    8.6% mean improvement (range: -7.7% – 16.5%)
          separability:    0.6% mean improvement (range: 0.0% – 1.2%)

=== PR ===
           constraints:   52.3% mean improvement (range: 32.9% – 65.3%)
                  size:   45.0% m